In [6]:
import sys
from pathlib import Path
from datetime import datetime, timedelta
from dotenv import load_dotenv
import time


from tradingbot.core.sequence import sequence_builder
from tradingbot.core.candles import candle_builder
from tradingbot.core.constants import Interval
from tradingbot.indicators import (
    RelativeStrengthIndex,
    StochasticOscillator,
    StochasticRSI,
)
from tradingbot.kite import (
    KiteSessionManager,
)
from tradingbot.core.ticker import Ticker


# Resolve paths so this notebook works whether the kernel starts in the repo root or notebooks/.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_HISTORIC_DATA_DIR = PROJECT_ROOT / "data" / "historic_data"
INDICATOR_DATA_DIR = PROJECT_ROOT / "data" / "historic_data_with_indicators"
RAW_HISTORIC_DATA_DIR.mkdir(parents=True, exist_ok=True)
INDICATOR_DATA_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv()
session_manager = KiteSessionManager()
session = session_manager.start_session(cli=False)



🔍 Checking for cached access token...
📖 Loaded cached access token from disk
✓ Access token validation successful
✅ Cached access token is valid!



In [7]:
STOCKS = [
    "RELIANCE",
    "HDFCBANK",
    "ICICIBANK",
    "INFY",
    "TCS",
    "SBIN",
    "AXISBANK",
    "KOTAKBANK",
    "LT",
    "ITC",
    "HINDUNILVR",
    "BHARTIARTL",
    "MARUTI",
    "M&M",
    "SUNPHARMA",
    "HCLTECH",
    "TECHM",
    "WIPRO",
    "TITAN",
    "ASIANPAINT",
    "BAJFINANCE",
    "BAJAJFINSV",
    "ULTRACEMCO",
    "NTPC",
    "POWERGRID",
    "ONGC",
    "COALINDIA",
    "TATASTEEL",
    "JSWSTEEL",
    "HINDALCO",
    "GRASIM",
    "ADANIPORTS",
    "DRREDDY",
    "CIPLA",
    "EICHERMOT",
    "HEROMOTOCO",
    "BAJAJ-AUTO",
    "NESTLEIND",
]

In [ ]:
# Download historic data with resume/skip support
import csv
from tradingbot.core.ticker import HISTORIC_INCEPTION_START

INTERVALS = ["1m", "5m", "15m", "1h", "day"]
SLEEP_SECONDS = 1
HISTORIC_START_LABEL = HISTORIC_INCEPTION_START.isoformat().replace(":", "-").replace("+", "plus")


def historic_csv_path(symbol, interval):
    normalized_interval = Interval.normalize(interval)
    return RAW_HISTORIC_DATA_DIR / f"{symbol.upper()}_{normalized_interval}_{HISTORIC_START_LABEL}_now.csv"


def is_complete_historic_csv(path):
    if not path.exists() or not path.is_file() or path.stat().st_size == 0:
        return False

    with path.open("r", newline="") as file_obj:
        reader = csv.DictReader(file_obj)
        required_fieldnames = Ticker.historic_candle_fieldnames()
        if not reader.fieldnames or any(field not in reader.fieldnames for field in required_fieldnames):
            return False
        return next(reader, None) is not None

if "ticker" in globals() and ticker is not None:
    try:
        ticker.websocket_client.close()
    except Exception:
        pass

ticker = None

downloaded_paths = {}
skipped_paths = {}
failed_downloads = {}

for index, symbol in enumerate(STOCKS, start=1):
    downloaded_paths[symbol] = {}
    skipped_paths[symbol] = {}
    failed_downloads[symbol] = {}

    for interval in INTERVALS:
        existing_path = historic_csv_path(symbol, interval)

        if is_complete_historic_csv(existing_path):
            skipped_paths[symbol][interval] = existing_path
            print(
                f"[{index}/{len(STOCKS)}] Skipping {symbol} "
                f"{interval}; already exists -> {existing_path}"
            )
            continue

        print(
            f"[{index}/{len(STOCKS)}] Fetching {symbol} "
            f"{interval} candles since inception..."
        )
        try:
            if ticker is None:
                ticker = Ticker(symbol=symbol, timeframe=interval, session=session)

            path = ticker.get_historic_data(
                ticker_name=symbol,
                timeframe=interval,
                csv_path=existing_path,
            )
            downloaded_paths[symbol][interval] = path
            print(f"    saved -> {path}")
        except Exception as exc:
            failed_downloads[symbol][interval] = exc
            print(f"    failed -> {exc}")

        time.sleep(SLEEP_SECONDS)

    if not downloaded_paths[symbol]:
        del downloaded_paths[symbol]

    if not skipped_paths[symbol]:
        del skipped_paths[symbol]

    if not failed_downloads[symbol]:
        del failed_downloads[symbol]

if ticker is not None:
    try:
        ticker.websocket_client.close()
    except Exception:
        pass

downloaded_paths, skipped_paths, failed_downloads


[1/38] Skipping RELIANCE 1m; already exists -> /Users/akash/PycharmProjects/AlgoTrade/data/historic_data/RELIANCE_minute_1990-01-01T00-00-00plus05-30_now.csv
[1/38] Skipping RELIANCE 5m; already exists -> /Users/akash/PycharmProjects/AlgoTrade/data/historic_data/RELIANCE_5minute_1990-01-01T00-00-00plus05-30_now.csv
[1/38] Skipping RELIANCE 15m; already exists -> /Users/akash/PycharmProjects/AlgoTrade/data/historic_data/RELIANCE_15minute_1990-01-01T00-00-00plus05-30_now.csv
[1/38] Skipping RELIANCE 1h; already exists -> /Users/akash/PycharmProjects/AlgoTrade/data/historic_data/RELIANCE_60minute_1990-01-01T00-00-00plus05-30_now.csv
[1/38] Skipping RELIANCE day; already exists -> /Users/akash/PycharmProjects/AlgoTrade/data/historic_data/RELIANCE_day_1990-01-01T00-00-00plus05-30_now.csv
[2/38] Skipping HDFCBANK 1m; already exists -> /Users/akash/PycharmProjects/AlgoTrade/data/historic_data/HDFCBANK_minute_1990-01-01T00-00-00plus05-30_now.csv
[2/38] Skipping HDFCBANK 5m; already exists -> /

Connection closed: None - None


Websocket closed: None - None
Connected websocket for BHARTIARTL: {"peer": "tcp4:35.154.247.7:443", "headers": {"date": "Fri, 15 May 2026 12:16:19 GMT", "connection": "upgrade", "upgrade": "websocket", "sec-websocket-accept": "Erb+WtzqrioTKBc9OQibeUsiecY="}, "version": 18, "protocol": null, "extensions": []}
    saved -> /Users/akash/PycharmProjects/AlgoTrade/data/historic_data/BHARTIARTL_minute_1990-01-01T00-00-00plus05-30_now.csv
[12/38] Fetching BHARTIARTL 5m candles since inception...
    saved -> /Users/akash/PycharmProjects/AlgoTrade/data/historic_data/BHARTIARTL_5minute_1990-01-01T00-00-00plus05-30_now.csv
[12/38] Fetching BHARTIARTL 15m candles since inception...
    saved -> /Users/akash/PycharmProjects/AlgoTrade/data/historic_data/BHARTIARTL_15minute_1990-01-01T00-00-00plus05-30_now.csv
[12/38] Fetching BHARTIARTL 1h candles since inception...
    saved -> /Users/akash/PycharmProjects/AlgoTrade/data/historic_data/BHARTIARTL_60minute_1990-01-01T00-00-00plus05-30_now.csv
[12/38

({'BHARTIARTL': {'1m': PosixPath('/Users/akash/PycharmProjects/AlgoTrade/data/historic_data/BHARTIARTL_minute_1990-01-01T00-00-00plus05-30_now.csv'),
   '5m': PosixPath('/Users/akash/PycharmProjects/AlgoTrade/data/historic_data/BHARTIARTL_5minute_1990-01-01T00-00-00plus05-30_now.csv'),
   '15m': PosixPath('/Users/akash/PycharmProjects/AlgoTrade/data/historic_data/BHARTIARTL_15minute_1990-01-01T00-00-00plus05-30_now.csv'),
   '1h': PosixPath('/Users/akash/PycharmProjects/AlgoTrade/data/historic_data/BHARTIARTL_60minute_1990-01-01T00-00-00plus05-30_now.csv'),
   'day': PosixPath('/Users/akash/PycharmProjects/AlgoTrade/data/historic_data/BHARTIARTL_day_1990-01-01T00-00-00plus05-30_now.csv')},
  'MARUTI': {'1m': PosixPath('/Users/akash/PycharmProjects/AlgoTrade/data/historic_data/MARUTI_minute_1990-01-01T00-00-00plus05-30_now.csv'),
   '5m': PosixPath('/Users/akash/PycharmProjects/AlgoTrade/data/historic_data/MARUTI_5minute_1990-01-01T00-00-00plus05-30_now.csv'),
   '15m': PosixPath('/User

Connection error: 1006 - connection was closed uncleanly (WebSocket closing handshake timeout (peer did not finish the opening handshake in time))


Websocket error: 1006 - connection was closed uncleanly (WebSocket closing handshake timeout (peer did not finish the opening handshake in time))


Connection closed: 1006 - connection was closed uncleanly (WebSocket closing handshake timeout (peer did not finish the opening handshake in time))


Websocket closed: 1006 - connection was closed uncleanly (WebSocket closing handshake timeout (peer did not finish the opening handshake in time))
